In [17]:
import pandas as pd
from pathlib import Path

In [18]:
root = Path.cwd()

In [19]:
if not (root / "data" / "processed").exists():
    if (root.parent / "data" / "processed").exists():
        root = root.parent
    else:
        raise FileNotFoundError(
            "Cannot find the data/processed folder."
        )


In [20]:
processed_dir = root / "data" / "processed"

In [21]:
# Load the three cleaned datasets
recommended = pd.read_csv(
    processed_dir / "works_recommended_cleaned.csv",
    dtype={"work_id": "string"}
)

sanctioned = pd.read_csv(
    processed_dir / "works_sanctioned_cleaned.csv",
    dtype={"work_id": "string"}
)

completed = pd.read_csv(
    processed_dir / "works_completed_cleaned.csv",
    dtype={"work_id": "string"}
)

In [23]:
print("All three datasets loaded successfully!")
print("Recommended:", recommended.shape)
print("Sanctioned:", sanctioned.shape)
print("Completed:", completed.shape)

All three datasets loaded successfully!
Recommended: (5000, 12)
Sanctioned: (36000, 14)
Completed: (16000, 13)


In [25]:
for name, df in {
    "recommended": recommended,
    "sanctioned": sanctioned,
    "completed": completed
}.items():

    print(f"\n{'=' * 40}")
    print(f"Dataset: {name}")
    print(f"Total records: {len(df)}")

    # Missing Work IDs
    missing = df["work_id"].isna()
    print("Missing Work IDs:", missing.sum())

    # Duplicate non-missing Work IDs
    valid_ids = df.loc[~missing, "work_id"]
    duplicate_mask = valid_ids.duplicated(keep=False)

    print(
        "Records with repeated valid Work IDs:",
        duplicate_mask.sum()
    )
    print(
        "Distinct repeated Work IDs:",
        valid_ids[duplicate_mask].nunique()
    )

    if duplicate_mask.any():
        print("\nExamples of repeated Work IDs:")
        print(
            df.loc[valid_ids[duplicate_mask].index]
            [["work_id"]]
            .head(10)
        )


Dataset: recommended
Total records: 5000
Missing Work IDs: 59
Records with repeated valid Work IDs: 0
Distinct repeated Work IDs: 0

Dataset: sanctioned
Total records: 36000
Missing Work IDs: 0
Records with repeated valid Work IDs: 0
Distinct repeated Work IDs: 0

Dataset: completed
Total records: 16000
Missing Work IDs: 0
Records with repeated valid Work IDs: 0
Distinct repeated Work IDs: 0


In [26]:
# Check missing Work IDs in all three datasets

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Total rows:", len(df))
    print("Missing Work IDs:", df["work_id"].isna().sum())
    print("Duplicate valid Work IDs:",
          df.loc[df["work_id"].notna(), "work_id"].duplicated().sum())


RECOMMENDED
Total rows: 5000
Missing Work IDs: 0
Duplicate valid Work IDs: 58

SANCTIONED
Total rows: 36000
Missing Work IDs: 0
Duplicate valid Work IDs: 0

COMPLETED
Total rows: 16000
Missing Work IDs: 0
Duplicate valid Work IDs: 0


In [27]:
# Extract records with missing Work IDs
missing_recommended = recommended[
    recommended["work_id"].isna()
].copy()

print("Records without Work IDs:", len(missing_recommended))

display(missing_recommended.head(10))

Records without Work IDs: 59


,sr_no,work_category,state,ida,hon_ble_members_of_parliament,constituency,work_description,recommended_date,recommended_amount,sanction_date,work_id,work_detail_from_reference
231,232,Normal/Others,Bihar,PURBI CHAMPARAN(DISTRICT PLANNING OFFICER MOTH...,Radha Mohan Singh,PURVI CHAMPARAN,MOTIHARI NAGAR KE MAHATMA GANDHI MARG ME CHARK...,2024-08-05,1497500.0,NaN,<NA>,NaN
232,233,Normal/Others,Bihar,PURBI CHAMPARAN(DISTRICT PLANNING OFFICER MOTH...,Radha Mohan Singh,PURVI CHAMPARAN,MOTIHARI NAGAR KE MAHATMA GANDHI MARG ME CHARK...,2024-08-05,1495500.0,NaN,<NA>,NaN
242,243,Normal/Others,Haryana,ROHTAK(DEPUTY COMMISSIONER ROHTAK_IDA),DEEPENDER SINGH HOODA,ROHTAK,Development of Works in Vishwakarma Panchal Dh...,2024-08-05,1100000.0,NaN,<NA>,NaN
440,441,Normal/Others,West Bengal,DARJEELING(DISTRICT MAGISTRATE DARJEELING_IDA),Raju Bista,DARJEELING,Construction of CC road at Nichu Line Lohagarh...,2024-08-16,500000.0,NaN,<NA>,NaN
508,509,Normal/Others,West Bengal,DARJEELING(DISTRICT MAGISTRATE DARJEELING_IDA),Raju Bista,DARJEELING,Construction of additional rooms at Milan Sang...,2024-08-16,500000.0,NaN,<NA>,NaN
558,559,Repair and Renovation,Kerala,ERNAKULAM(DISTRICT COLLECTOR ERNAKULAM_IDA),Shri Hibi Eden,ERNAKULAM,"Renovation of Kottakunnu Road at Ward 21, Eloo...",2024-08-16,1500000.0,NaN,<NA>,NaN
920,921,Normal/Others,Andhra Pradesh,Eluru(DISTRICT COLLECTOR ELURU_IDA),Putta Mahesh Kumar,ELURU,Providing Laboratory equipment to Physical Sci...,2024-08-26,400000.0,NaN,<NA>,NaN
1417,1418,Normal/Others,Rajasthan,ALWAR(DISTRICT COLLECTOR ALWAR_IDA),SANJNA JATAV,BHARATPUR(SC),CC interlocking v Soundarya Nirman karya Suraj...,2024-09-01,100000.0,NaN,<NA>,NaN
1572,1573,Normal/Others,Tamil Nadu,TIRUPPUR(DISTRICT COLLECTOR TIRUPPUR_IDA),K Subbarayan,TIRUPPUR,Providing new borewell and water supply facili...,2024-09-03,800000.0,NaN,<NA>,NaN
1748,1749,Normal/Others,West Bengal,North 24 Parganas(DISTRICT MAGISTRATE NORTH TW...,SK NURUL ISLAM,BASIRHAT,Installation of 20 nos. Street LED Electric Li...,2024-09-05,440000.0,NaN,<NA>,NaN


In [28]:
print(recommended.columns.tolist())

['sr_no', 'work_category', 'state', 'ida', 'hon_ble_members_of_parliament', 'constituency', 'work_description', 'recommended_date', 'recommended_amount', 'sanction_date', 'work_id', 'work_detail_from_reference']


In [29]:
# Preserve records that cannot be linked
recommended_missing_id = recommended[
    recommended["work_id"].isna()
].copy()

# Keep only records with valid IDs for integration
recommended_for_merge = recommended[
    recommended["work_id"].notna()
].copy()

print("Records retained for merging:", len(recommended_for_merge))
print("Records requiring review:", len(recommended_missing_id))

# Check duplicate valid IDs
print(
    "Duplicate valid IDs:",
    recommended_for_merge["work_id"].duplicated().sum()
)

Records retained for merging: 4941
Records requiring review: 59
Duplicate valid IDs: 0


In [32]:
# Keep the source-specific columns separate
def rename_source_columns(df, source):
    return df.rename(columns={
        col: f"{source}__{col}"
        for col in df.columns
        if col != "work_id"
    })


recommended_table = rename_source_columns(
    recommended_for_merge, "recommended"
)

sanctioned_table = rename_source_columns(
    sanctioned, "sanctioned"
)

completed_table = rename_source_columns(
    completed, "completed"
)

# Merge Recommended + Sanctioned
master = recommended_table.merge(
    sanctioned_table,
    on="work_id",
    how="outer",
    validate="one_to_one",
    indicator="recommended_sanctioned_merge"
)

# Add Completed
master = master.merge(
    completed_table,
    on="work_id",
    how="outer",
    validate="one_to_one",
    indicator="completed_merge"
)

print("Master dataset shape:", master.shape)
print("Distinct Work IDs:", master["work_id"].nunique())

display(master.head())

Master dataset shape: (37326, 39)
Distinct Work IDs: 37326


,recommended__sr_no,recommended__work_category,recommended__state,recommended__ida,recommended__hon_ble_members_of_parliament,recommended__constituency,recommended__work_description,recommended__recommended_date,recommended__recommended_amount,recommended__sanction_date,...,completed__ida,completed__work_description,completed__hon_ble_members_of_parliament,completed__constituency,completed__image,completed__completion_date,completed__amount_disbursed,completed__work_detail_from_reference,completed__image_marker_present,completed_merge
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Ahmedabad(DISTRICT COLLECTOR AHMADABAD_IDA),Construction of C.C Road at Bhalan Para Vistar...,Devusinh Jesingbhai Chauhan,KHEDA,Images,2025-04-15,450000.0,"Construction of roads, link roads, pathways or...",1.0,both
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KHEDA(DISTRICT COLLECTOR KHEDA_IDA),Construction of C.C Road Work At Main Road tow...,Devusinh Jesingbhai Chauhan,KHEDA,NaN,2025-05-29,250000.0,"Construction of roads, link roads, pathways or...",0.0,both
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KHEDA(DISTRICT COLLECTOR KHEDA_IDA),Construction Of Paver Block Work at BHaratpura...,Devusinh Jesingbhai Chauhan,KHEDA,NaN,2025-05-06,100000.0,"Construction of roads, link roads, pathways or...",0.0,both
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KHEDA(DISTRICT COLLECTOR KHEDA_IDA),Construction Of Paver Block Work at Bharatpura...,Devusinh Jesingbhai Chauhan,KHEDA,NaN,2025-05-06,50000.0,"Construction of roads, link roads, pathways or...",0.0,both


In [34]:
print("Total rows:", len(master))
print("Unique Work IDs:", master["work_id"].nunique())
print("Duplicate Work IDs:", master["work_id"].duplicated().sum())
print("Missing Work IDs:", master["work_id"].isna().sum())

print("\nRecommended–Sanctioned merge status:")
print(master["recommended_sanctioned_merge"].value_counts(
    dropna=False
))

print("\nCompleted merge status:")
print(master["completed_merge"].value_counts(
    dropna=False
))

Total rows: 37326
Unique Work IDs: 37326
Duplicate Work IDs: 0
Missing Work IDs: 0

Recommended–Sanctioned merge status:
recommended_sanctioned_merge
right_only    31059
both           4941
NaN            1326
left_only         0
Name: count, dtype: int64

Completed merge status:
completed_merge
left_only     21326
both          14674
right_only     1326
Name: count, dtype: int64


In [36]:
recommended_missing_id.to_csv(
    "../data/processed/recommended_missing_work_ids.csv",
    index=False
)

master.to_csv("../data/processed/mplads_master_dataset.csv", index=False)

In [37]:

# 1. Inspect merge-status columns
status_columns = [
    "recommended_sanctioned_merge",
    "completed_merge"
]

for col in status_columns:
    print(f"\n--- {col} ---")

    if col in master.columns:
        print(master[col].value_counts(dropna=False))
        print("Missing values:", master[col].isna().sum())
    else:
        print(f"Column '{col}' not found.")

# 2. Inspect rows with missing first-stage status
if "recommended_sanctioned_merge" in master.columns:
    missing_status = master[
        master["recommended_sanctioned_merge"].isna()
    ]

    print("\nRows with missing first-stage status:",
          len(missing_status))

    display(missing_status.head(10))



--- recommended_sanctioned_merge ---
recommended_sanctioned_merge
right_only    31059
both           4941
NaN            1326
left_only         0
Name: count, dtype: int64
Missing values: 1326

--- completed_merge ---
completed_merge
left_only     21326
both          14674
right_only     1326
Name: count, dtype: int64
Missing values: 0

Rows with missing first-stage status: 1326


,recommended__sr_no,recommended__work_category,recommended__state,recommended__ida,recommended__hon_ble_members_of_parliament,recommended__constituency,recommended__work_description,recommended__recommended_date,recommended__recommended_amount,recommended__sanction_date,...,completed__ida,completed__work_description,completed__hon_ble_members_of_parliament,completed__constituency,completed__image,completed__completion_date,completed__amount_disbursed,completed__work_detail_from_reference,completed__image_marker_present,completed_merge
167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KHEDA(DISTRICT COLLECTOR KHEDA_IDA),Construction of Paver Block Work at Mahamdpura...,Devusinh Jesingbhai Chauhan,KHEDA,NaN,2025-10-04,198587.0,"Construction of roads, link roads, pathways or...",0.0,right_only
493,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Alluri Sitharama Raju(DISTRICT COLLECTOR ALLUR...,Providing CC road from 10-172 to 8-4 at Gudiva...,Gumma Thanuja Rani,ARAKU(ST),Images,2025-09-07,998297.0,"Construction of roads, link roads, pathways or...",1.0,right_only
820,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source a...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199939.0,Installing tube-wells and borewells,1.0,right_only
821,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source i...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199955.0,Installing tube-wells and borewells,1.0,right_only
822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source o...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199960.0,Installing tube-wells and borewells,1.0,right_only
823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source a...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199998.0,Installing tube-wells and borewells,1.0,right_only
824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source n...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199939.0,Installing tube-wells and borewells,1.0,right_only
825,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source n...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199947.0,Installing tube-wells and borewells,1.0,right_only
826,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source n...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199960.0,Installing tube-wells and borewells,1.0,right_only
827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,KURNOOL(DISTRICT COLLECTOR KURNOOL_IDA),Drilling of a new borewell for scheme source n...,BASTIPATI NAGARAJU PANCHALINGALA,KURNOOL,Images,2026-01-09,199955.0,Installing tube-wells and borewells,1.0,right_only


In [38]:

# Verify the merge-status relationship
completed_only = master[
    master["completed_merge"].eq("right_only")
]

print("Completed-only records:", len(completed_only))

print(
    "Missing first-stage status among completed-only records:",
    completed_only["recommended_sanctioned_merge"].isna().sum()
)

print(
    "Missing first-stage status outside completed-only records:",
    master.loc[
        master["completed_merge"].ne("right_only"),
        "recommended_sanctioned_merge"
    ].isna().sum()
)


Completed-only records: 1326
Missing first-stage status among completed-only records: 1326
Missing first-stage status outside completed-only records: 0


In [39]:

print("SOURCE DATASET COUNTS")
print("Recommended:", len(recommended_for_merge))
print("Sanctioned:", len(sanctioned))
print("Completed:", len(completed))

print("\nMASTER DATASET")
print("Total rows:", len(master))
print("Unique Work IDs:", master["work_id"].nunique())

print("\nMATCHING COVERAGE")
print(
    "Recommended–Sanctioned matched:",
    master["recommended_sanctioned_merge"].eq("both").sum()
)
print(
    "Completed matched to earlier datasets:",
    master["completed_merge"].eq("both").sum()
)


SOURCE DATASET COUNTS
Recommended: 4941
Sanctioned: 36000
Completed: 16000

MASTER DATASET
Total rows: 37326
Unique Work IDs: 37326

MATCHING COVERAGE
Recommended–Sanctioned matched: 4941
Completed matched to earlier datasets: 14674


In [40]:
# Overall dataset quality
quality_report = pd.DataFrame({
    "column": master.columns,
    "data_type": [
        str(master[col].dtype) for col in master.columns
    ],
    "non_null_count": [
        master[col].notna().sum() for col in master.columns
    ],
    "missing_count": [
        master[col].isna().sum() for col in master.columns
    ],
    "missing_percentage": [
        round(master[col].isna().mean() * 100, 2)
        for col in master.columns
    ],
    "unique_values": [
        master[col].nunique() for col in master.columns
    ]
})

print("Master dataset shape:", master.shape)
print("Duplicate Work IDs:", master["work_id"].duplicated().sum())
print("Missing Work IDs:", master["work_id"].isna().sum())

display(
    quality_report.sort_values(
        "missing_percentage", ascending=False
    )
)


Master dataset shape: (37326, 39)
Duplicate Work IDs: 0
Missing Work IDs: 0


,column,data_type,non_null_count,missing_count,missing_percentage,unique_values
6,recommended__work_description,str,4927,32399,86.80,4350
0,recommended__sr_no,float64,4941,32385,86.76,4941
11,recommended__work_detail_from_reference,str,4941,32385,86.76,74
9,recommended__sanction_date,str,4941,32385,86.76,379
1,recommended__work_category,str,4941,32385,86.76,3
7,recommended__recommended_date,str,4941,32385,86.76,92
8,recommended__recommended_amount,float64,4941,32385,86.76,1639
5,recommended__constituency,str,4941,32385,86.76,175
4,recommended__hon_ble_members_of_parliament,str,4941,32385,86.76,175
3,recommended__ida,str,4941,32385,86.76,230


In [41]:
quality_report.to_csv(
    "../reports/mplads_data_quality_report.csv", index=False
)

print("Master dataset and data quality report exported.")

Master dataset and data quality report exported.
